In [ ]:
def validate_numbers(repository):
    stars = repository['stars']
    forks = repository['forks']
    watchers = repository['watchers']
    languageCount = repository['languageCount']

    if (stars < 0 or forks < 0 or watchers < 0 or languageCount < 0):
        return False
    return True

In [ ]:
from datetime import date
def validate_date(repository):
    createdAt = repository['createdAt']
    pushedAt = repository['pushedAt']

    if not isinstance(createdAt,date):
        return False
    if not isinstance(pushedAt,date):
        return False
    if repository['repo_age_days'] < 0:
        return False
    return True

In [ ]:
def validate_name(repository):
    name = repository['name']
    if pd.isna(name):
        return False
    if not isinstance(name,str):
        return False
    if name.strip() == "":
        return False
    return True

In [66]:
#Function 1
import pandas as pd
def load_data(file_path):
    return pd.read_csv(file_path)

In [65]:
#Function 2
def convert_dates(df):
    df['pushedAt'] = pd.to_datetime(df['pushedAt'], utc=True)
    df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)
    return df

In [51]:
def remove_empty_names(df):
    df = df[df['name'].fillna('').str.strip() != '']
    return df

In [68]:
def remove_duplicates(df):
    df = df.drop_duplicates(subset=['owner', 'name'])
    return df

In [55]:
def calculate_repo_age(df):
    df['repo_age_days'] = (
        df['pushedAt'] - df['createdAt']
    ).dt.days
    return df

In [54]:
def remove_invalid_repo_age(df):
    df = df[df['repo_age_days'] >= 0]
    return df

In [61]:
import ast

def combine_text(df):
    def combine_row(row):
        description = row['description'] if pd.notna(row['description']) else ''
        primary_language = row['primaryLanguage'] if pd.notna(row['primaryLanguage']) else ''

        languages = ast.literal_eval(row['languages']) if row['languages'] else []
        topics = ast.literal_eval(row['topics']) if row['topics'] else []

        return ' '.join([
            str(row['name']),
            str(description),
            str(primary_language),
            *languages,
            *topics
        ])

    df['combined_text'] = df.apply(combine_row, axis=1)

    return df


In [60]:
df.info()

<class 'pandas.DataFrame'>
Index: 99949 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   owner            99949 non-null  str                
 1   name             99949 non-null  str                
 2   stars            99949 non-null  int64              
 3   forks            99949 non-null  int64              
 4   watchers         99949 non-null  int64              
 5   languageCount    99949 non-null  int64              
 6   description      97160 non-null  str                
 7   primaryLanguage  91733 non-null  str                
 8   createdAt        99949 non-null  datetime64[us, UTC]
 9   pushedAt         99949 non-null  datetime64[us, UTC]
 10  languages        99949 non-null  str                
 11  topics           99949 non-null  str                
 12  isFork           99949 non-null  bool               
 13  isArchived       99949 non-null 

In [ ]:
def process_data(file_path):
    df = load_data(file_path)
    df = convert_dates(df)
    df = calculate_repo_age(df)
    df = remove_invalid_repo_age(df)
    df = remove_empty_names(df)
    df = remove_duplicates(df)
    df = combine_text(df)

    return df

In [69]:
df = process_data('../data/processed_v2/repositories_01.csv')
df.head(3)

,owner,name,stars,forks,watchers,languageCount,description,primaryLanguage,createdAt,pushedAt,languages,topics,isFork,isArchived,repo_age_days,combined_text
0,freeCodeCamp,freeCodeCamp,426893,41377,8588,6,freeCodeCamp.org's open-source codebase and cu...,TypeScript,2014-12-24 17:49:19+00:00,2025-08-31 09:33:14+00:00,"['TypeScript', 'JavaScript', 'CSS', 'Dockerfil...","['learn-to-code', 'nonprofits', 'programming',...",False,False,3902,freeCodeCamp freeCodeCamp.org's open-source co...
1,codecrafters-io,build-your-own-x,415801,38976,6247,1,Master programming by recreating your favorite...,Markdown,2018-05-09 12:03:18+00:00,2025-08-29 00:08:20+00:00,['Markdown'],"['programming', 'tutorials', 'tutorial-code', ...",False,False,2668,build-your-own-x Master programming by recreat...
2,sindresorhus,awesome,396445,31414,8011,0,😎 Awesome lists about all kinds of interesting...,NaN,2014-07-11 13:42:37+00:00,2025-07-18 18:37:33+00:00,[],"['awesome', 'awesome-list', 'unicorns', 'lists...",False,False,4025,awesome 😎 Awesome lists about all kinds of int...
